In [1]:
import os
import shutil
from PIL import Image

# Define root folder where all datasets live
DATASET_ROOT = "all_datasets"

# Define class groups and label IDs
class_groups = {
    'blood': ['blood1'],
    'face_mask': ['mask1'],
    'gun': ['gun1'],
    'knife':['knife1','knife2','knife3']
}
class_ids = {cls: idx for idx, cls in enumerate(class_groups)}

# Define splits to handle
splits = ['train', 'valid', 'test']

# Output directory
output_base = os.path.join(DATASET_ROOT,'merged_dataset')

# Create output folders
for split in splits:
    os.makedirs(os.path.join(output_base, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_base, 'labels', split), exist_ok=True)

# Image extension whitelist
valid_exts = ('.jpg', '.jpeg', '.png')

In [2]:
# Function to copy and sanitize files
def process_all():
    for class_name, folders in class_groups.items():
        class_id = class_ids[class_name]
        for folder in folders:
            for split in splits:
                image_dir = os.path.join(DATASET_ROOT, folder, split, 'images')
                label_dir = os.path.join(DATASET_ROOT, folder, split, 'labels')

                if not os.path.exists(image_dir) or not os.path.exists(label_dir):
                    print(f"[Warning] Missing path: {image_dir} or {label_dir}")
                    continue

                for filename in os.listdir(image_dir):
                    if not filename.lower().endswith(valid_exts):
                        continue

                    name, ext = os.path.splitext(filename)
                    unique_name = f"{folder}_{name}"  # Ensure unique filenames

                    src_img = os.path.join(image_dir, filename)
                    dst_img = os.path.join(output_base, 'images', split, unique_name + ext)
                    dst_lbl = os.path.join(output_base, 'labels', split, unique_name + ".txt")
                    src_lbl = os.path.join(label_dir, name + ".txt")

                    # Check image integrity
                    try:
                        with Image.open(src_img) as img:
                            img.verify()
                    except Exception as e:
                        print(f"[Corrupt Image Skipped] {src_img} — {e}")
                        continue  # Skip corrupted image and label

                    # Copy image
                    shutil.copyfile(src_img, dst_img)

                    # Process label file
                    if os.path.exists(src_lbl):
                        new_lines = []
                        with open(src_lbl, 'r') as f:
                            for line in f:
                                parts = line.strip().split()
                                if len(parts) != 5:
                                    continue  # Skip malformed lines
                                parts[0] = str(class_id)
                                new_lines.append(" ".join(parts))

                        if new_lines:
                            with open(dst_lbl, 'w') as f:
                                f.write("\n".join(new_lines))

# Run the merge
process_all()

# Write data.yaml file
yaml_path = os.path.join(output_base, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(f"""train: {output_base}/images/train
val: {output_base}/images/val
test: {output_base}/images/test

nc: 4
names: ['blood', 'face_mask', 'gun', 'knife']
""")

print("\n✅ Dataset merge complete! Saved to 'merged_dataset/' with clean labels.")

[Warning] Missing path: all_datasets\mask1\test\images or all_datasets\mask1\test\labels

✅ Dataset merge complete! Saved to 'merged_dataset/' with clean labels.


In [3]:
import os
import shutil
import random
from PIL import Image

# Root directory where all dataset folders (fire1, flood2, etc.) live
DATASET_ROOT = "all_datasets"

# Define class grouping and class IDs
class_groups = {
    'blood': ['blood1'],
    'face_mask': ['mask1'],
    'gun': ['gun1'],
    'knife':['knife1','knife2','knife3']
}
class_ids = {cls: idx for idx, cls in enumerate(class_groups)}
splits = ['train', 'valid', 'test']

# Output directory for merged dataset
output_base = os.path.join(DATASET_ROOT, 'merged_dataset_new')
for split in splits:
    os.makedirs(os.path.join(output_base, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_base, 'labels', split), exist_ok=True)

valid_exts = ('.jpg', '.jpeg', '.png')

# Helper to check image validity
def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except:
        return False

# Step 1: Gather all valid image-label pairs per class
collected_data = {cls: [] for cls in class_groups}

for class_name, folders in class_groups.items():
    for folder in folders:
        for split in splits:
            img_dir = os.path.join(DATASET_ROOT, folder, split, 'images')
            lbl_dir = os.path.join(DATASET_ROOT, folder, split, 'labels')

            if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
                print(f"[Warning] Missing {img_dir} or {lbl_dir}")
                continue

            for file in os.listdir(img_dir):
                if not file.lower().endswith(valid_exts):
                    continue
                name, ext = os.path.splitext(file)
                img_path = os.path.join(img_dir, file)
                lbl_path = os.path.join(lbl_dir, name + ".txt")

                if not os.path.exists(lbl_path):
                    continue
                if not is_valid_image(img_path):
                    print(f"[Corrupt Skipped] {img_path}")
                    continue

                unique_name = f"{folder}_{name}"
                collected_data[class_name].append((img_path, lbl_path, unique_name, ext, split))

# Step 2: Randomly select 500 entries per class and copy
for class_name, data in collected_data.items():
    class_id = class_ids[class_name]
    random.shuffle(data)
    selected = data[:500]  # Select up to 500

    for img_path, lbl_path, unique_name, ext, split in selected:
        dst_img = os.path.join(output_base, 'images', split, unique_name + ext)
        dst_lbl = os.path.join(output_base, 'labels', split, unique_name + ".txt")

        shutil.copyfile(img_path, dst_img)

        new_lines = []
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                parts[0] = str(class_id)
                new_lines.append(" ".join(parts))

        if new_lines:
            with open(dst_lbl, 'w') as f:
                f.write("\n".join(new_lines))

print("✅ Random 500 per class merged successfully!")

# Step 3: Write data.yaml
yaml_path = os.path.join(output_base, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(f"""train: {output_base}/images/train
val: {output_base}/images/valid
test: {output_base}/images/test

nc: 4
names: ['blood', 'face_mask', 'gun','knife']
""")
print("📄 data.yaml created.")

[Warning] Missing all_datasets\mask1\test\images or all_datasets\mask1\test\labels
✅ Random 500 per class merged successfully!
📄 data.yaml created.
